In [1]:
#this notebook establishes parameter sets that will be used by LASAM

In [2]:
#!pip install smt

In [3]:
import numpy as np
import random 
from datetime import datetime #used to record total runtime
# import os #needed to run system commands from python
import subprocess #needed to run system commands from python, more flexible and safer than os
from smt.sampling_methods import LHS #Latin hypercube sampling
import sys
import matplotlib.pyplot as plt
import re
import os

os.chdir('../../')

In [4]:
theta_r_range = [0.01 ,0.15]
theta_e_range = [0.3, 0.8]
log_alpha_range =   [-3.0, -0.52] #corresponding to 0.001 and 0.3, via log_10
n_range =       [1.01, 3] #I recommend n to be no smaller than 1.01. Weird things happen with the water retention curve below that  
log_Ks_range =      [-3, 2] #corresponding to 0.001 and 100, via log_10
initial_psi_range = [10, 3000]
with_ponded_head = [-1, 1] #later, ponded head will be set to zero or nonzero based on this
field_capacity_psi_range = [10.3, 516.5]


In [5]:
#establishes limits for Latin hypercube.
#in this case there's going to be a new hypercube for each of the three soil layers.

xlimits = np.array([[theta_r_range[0], theta_r_range[1]],
                    [theta_e_range[0], theta_e_range[1]],
                    [log_alpha_range[0], log_alpha_range[1]],
                    [n_range[0], n_range[1]],
                    [log_Ks_range[0], log_Ks_range[1]],
                    [initial_psi_range[0], initial_psi_range[1]],
                    [with_ponded_head[0], with_ponded_head[1]],
                    [field_capacity_psi_range[0], field_capacity_psi_range[1]],
                    ])


In [6]:
samplingx = LHS(xlimits=xlimits,random_state=3+11) #random_state is for reproducability
samplingy = LHS(xlimits=xlimits,random_state=4+11) #random_state is for reproducability
samplingz = LHS(xlimits=xlimits,random_state=5+11) #random_state is for reproducability

#this creates the Latin hypercube containing all parameter sets, where num is the number of parameter sets desired.
num = 10000
x = samplingx(num)
y = samplingy(num)
z = samplingz(num)

In [7]:
x

array([[ 6.6707000e-02,  5.9517500e-01, -2.3632600e+00, ...,
         1.1855185e+03,  9.6790000e-01,  4.8752005e+02],
       [ 2.7661000e-02,  3.0037500e-01, -1.1785640e+00, ...,
         1.0402045e+03, -7.0750000e-01,  2.6878050e+01],
       [ 8.6363000e-02,  4.1587500e-01, -5.3178000e-01, ...,
         1.6385035e+03, -5.3370000e-01,  2.0247883e+02],
       ...,
       [ 1.4736100e-01,  4.2972500e-01, -1.1222680e+00, ...,
         9.3465750e+02, -4.5390000e-01,  4.9789715e+02],
       [ 3.1707000e-02,  5.2862500e-01, -2.8662040e+00, ...,
         1.0159855e+03,  5.2410000e-01,  5.0062010e+01],
       [ 9.9089000e-02,  6.9557500e-01, -2.7365000e+00, ...,
         1.7607945e+03, -1.6510000e-01,  4.3841865e+02]])

In [8]:
y

array([[ 8.8477000e-02,  5.9187500e-01, -5.3872400e-01, ...,
         6.8977650e+02,  2.8010000e-01,  2.5815030e+01],
       [ 1.0397500e-01,  6.6012500e-01, -1.0114120e+00, ...,
         1.4330905e+03, -7.3350000e-01,  5.4212850e+01],
       [ 1.4820100e-01,  7.2457500e-01, -1.6991160e+00, ...,
         2.0059745e+03,  3.9670000e-01,  4.9217709e+02],
       ...,
       [ 1.1828300e-01,  7.7162500e-01, -1.6698520e+00, ...,
         2.9884885e+03, -2.8710000e-01,  4.0526255e+02],
       [ 8.2261000e-02,  6.0552500e-01, -2.1355960e+00, ...,
         2.1934475e+03, -3.0090000e-01,  7.0411250e+01],
       [ 1.3838700e-01,  6.6477500e-01, -2.4026920e+00, ...,
         2.4795905e+03,  2.2700000e-02,  4.6028649e+02]])

In [9]:
z

array([[ 1.0145500e-01,  7.6987500e-01, -2.7890760e+00, ...,
         1.1750535e+03, -6.7670000e-01,  2.9131693e+02],
       [ 6.5699000e-02,  6.0187500e-01, -2.3084520e+00, ...,
         2.2287295e+03, -9.1300000e-02,  1.5327619e+02],
       [ 4.5973000e-02,  4.6602500e-01, -2.5321480e+00, ...,
         1.8618565e+03, -8.4290000e-01,  4.3244549e+02],
       ...,
       [ 1.4235000e-02,  3.3562500e-01, -2.0969080e+00, ...,
         2.4763015e+03,  9.8370000e-01,  3.2614349e+02],
       [ 7.3343000e-02,  4.0082500e-01, -1.0815960e+00, ...,
         2.0349775e+03,  1.6850000e-01,  2.0075775e+02],
       [ 1.4356700e-01,  7.1322500e-01, -1.7640920e+00, ...,
         2.5549385e+03, -5.3830000e-01,  2.0101085e+02]])

In [10]:
num_runs = 0
num_successful_runs = 0
unsuccessful_parameter_sets = []
#error types are: mass balance, timeout due to infinite loop, and nonzero ponded head when its max is set to 0.
unsuccessful_parameter_sets_ponded_water = []
unsuccessful_parameter_sets_timeout = []
unsuccessful_parameter_sets_mass_bal = []
runtime_vec = []
global_balance_vec = []

begin_time = datetime.now() #used to record total runtime

# test_set = [734]
# for parameter_set in test_set:
# for parameter_set in range(test_set[0],num):

for parameter_set in range(num):

    x[parameter_set,2] = 10**x[parameter_set,2]
    x[parameter_set,4] = 10**x[parameter_set,4]
    y[parameter_set,2] = 10**y[parameter_set,2]
    y[parameter_set,4] = 10**y[parameter_set,4]
    z[parameter_set,2] = 10**z[parameter_set,2]
    z[parameter_set,4] = 10**z[parameter_set,4]
    print("parameter set:")
    print(parameter_set)
#     print("actual alpha:")
#     print(x[parameter_set,2])
#     print("actual Ks:")
#     print(x[parameter_set,4])
    with open('data/vG_default_params_LHS_Koptis_Farms_3_layer.dat', 'r') as file:
    # read a list of lines into data
        parameters_file_text = file.readlines()
        file.close()
#     layer_to_replace = random.randint(1, 3)
    layers_to_replace = [1,2,3]
#     print("layer_to_replace: \n")
#     print(layer_to_replace)
    for layer_to_replace in layers_to_replace:
        if layer_to_replace == 1:
            parameters_file_text[layer_to_replace] = '"Clay"\t                ' + str(x[parameter_set,0]) + '\t' + str(x[parameter_set,1]) + '\t' + str(x[parameter_set,2]) + '\t' + str(x[parameter_set,3]) + '\t' + str(x[parameter_set,4]) + '\n'    
        if layer_to_replace == 2:
            parameters_file_text[layer_to_replace] = '"Clay"\t                ' + str(y[parameter_set,0]) + '\t' + str(y[parameter_set,1]) + '\t' + str(y[parameter_set,2]) + '\t' + str(y[parameter_set,3]) + '\t' + str(y[parameter_set,4]) + '\n' 
        if layer_to_replace == 3:
            parameters_file_text[layer_to_replace] = '"Clay"\t                ' + str(z[parameter_set,0]) + '\t' + str(z[parameter_set,1]) + '\t' + str(z[parameter_set,2]) + '\t' + str(z[parameter_set,3]) + '\t' + str(z[parameter_set,4]) + '\n' 
            
    with open('data/vG_default_params_LHS_Koptis_Farms_3_layer.dat', 'w') as file:
        file.writelines( parameters_file_text )
        file.close()
        
    with open('configs/config_lasam_LHS_Koptis_Farms_1_layer.txt', 'r') as file:
    # read a list of lines into data
        config_file_text = file.readlines()
        file.close()
    config_file_text[4] = 'initial_psi=' + str(x[parameter_set,5]) + '[cm]\n'
    
    if (x[parameter_set,6]<0):
        config_file_text[8] = 'ponded_depth_max=0[cm]\n'
    else:
        config_file_text[8] = 'ponded_depth_max=2[cm]\n'
    if (x[parameter_set,6]>0.5):
        config_file_text[8] = 'ponded_depth_max=5[cm]\n'
        
    config_file_text[13] = 'field_capacity_psi=' + str(x[parameter_set,7]) + '[cm]\n'
        
    with open('configs/config_lasam_LHS_Koptis_Farms_1_layer.txt', 'w') as file:
        file.writelines( config_file_text )
        file.close()
        
    print("running LASAM ...")
#     result = !./build/lasam_standalone configs/config_lasam_LHS_Phillipsburg.txt
#     result = !./build/lasam_standalone configs/config_lasam_LHS_Koptis_Farms.txt 
    start_time_loc = datetime.now() #used to record local runtime
    result = !./build/lasam_standalone configs/config_lasam_LHS_Koptis_Farms_1_layer.txt 
    end_time_loc = datetime.now() #used to record local runtime
    runtime_loc = end_time_loc - start_time_loc
    runtime_vec.append(runtime_loc.total_seconds())
    
    for row in result:
        print(row)
    print(" ")
    print(" ")
    
    run_success = False
    surface_ponded_water = 0.0;
    if (len(result) > 1):
#         if (result[8][0:20]=='Surface ponded water'):
#             surface_ponded_water = float(result[8][30:38])
#         if ((result[2] == '-------------------- Simulation Summary ----------------- ') & (result[8]=='Surface ponded water      =   0.0000000000 cm')):
        if ('-------------------- Simulation Summary ----------------- ' in result):
            num_successful_runs = num_successful_runs + 1
            run_success = True

    if (run_success == False):
        unsuccessful_parameter_sets.append(parameter_set)
        print("unsuccessful parameter set: #################################################################################### ")
        print(parameter_set)
        print(" ")
        print(" ")
        print(" ")
        print(" ")
        print(" ")
        for line in parameters_file_text:
            print(line)
        print("adding code that causes the notebook to crash when an individual run fails, for testing purposes")
        sys.exit()
        if ('mass balance closure not possible within 100000 iterations. Timeout ' in result):
            unsuccessful_parameter_sets_timeout.append(parameter_set)
        if (len(result)>1):
            if (result[1]=='Local mass balance at this timestep... '):
                unsuccessful_parameter_sets_mass_bal.append(parameter_set)
                
    if (run_success):

        # Initialize the variable for the global balance number
        global_balance = None

        # Loop through the list to find the string containing "Global balance"
        for line in result:
            if "Global balance" in line:
                # Extract the number using a regular expression
                number_str = re.search(r"[-+]?\d*\.\d+e[-+]?\d+", line).group()
                # Convert the extracted string to a float
                global_balance = float(number_str)
                break  # Stop the loop once the number is found

#         print(global_balance)
        global_balance_vec.append(np.log10(abs(global_balance)))
            
            
        
    num_runs = num_runs + 1
    
end_time = datetime.now() #used to record total runtime
total_time = end_time - begin_time
print("all runs complete.")



    

parameter set:
0
running LASAM ...

********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  57.6384350498 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        = 184.8193895149 cm
Final water in soil       =  54.0533356348 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            =  17.1106104851 cm
GIUH runoff               =  17.1106104851 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  12.6692184047 cm
Total AET                 = 175.7352707335 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       =  17.1106104851 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   -2.082349e-07 cm
Time                      =   1.046 sec 
 
 
parameter set:
1
running LASAM ...

********************************************************* 
--


********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  43.5159732771 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        =  30.3497824416 cm
Final water in soil       =  34.3793246742 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            = 171.5802175584 cm
GIUH runoff               = 171.5802175584 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  -0.3060596182 cm
Total AET                 =  39.7924908142 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       = 171.5802175584 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   -1.514418e-07 cm
Time                      =   2.26 sec 
 
 
parameter set:
11
running LASAM ...

********************************************************* 
-------------------- Simulation Summa


********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  48.9236505739 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        = 174.0504906758 cm
Final water in soil       =  47.4131216160 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            =  27.8795093242 cm
GIUH runoff               =  27.8795093242 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  23.5708002557 cm
Total AET                 = 151.9902178497 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       =  27.8795093242 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   1.528305e-06 cm
Time                      =   1.477 sec 
 
 
parameter set:
21
running LASAM ...

********************************************************* 
-------------------- Simulation Summa


********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  14.1882419122 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        =  76.6968476271 cm
Final water in soil       =  20.2076317546 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            = 125.2331523729 cm
GIUH runoff               = 125.2331523729 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  -0.0011439990 cm
Total AET                 =  70.6786017923 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       = 125.2331523729 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   -8.631726e-09 cm
Time                      =   7.284 sec 
 
 
parameter set:
31
running LASAM ...

********************************************************* 
-------------------- Simulation Summ


********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  67.6828004492 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        = 201.9300000000 cm
Final water in soil       =  67.1118319319 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            =   0.0000000000 cm
GIUH runoff               =   0.0000000000 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  27.2168516035 cm
Total AET                 = 175.2841169294 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       =   0.0000000000 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   -1.560464e-08 cm
Time                      =   0.78 sec 
 
 
parameter set:
41
running LASAM ...

********************************************************* 
-------------------- Simulation Summa

...


********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  51.8510252035 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        = 201.8064386330 cm
Final water in soil       =  51.6142086619 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            =   0.1235613670 cm
GIUH runoff               =   0.1235613670 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  31.9541397062 cm
Total AET                 = 170.0891154681 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       =   0.1235613670 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   3.419487e-10 cm
Time                      =   1.063 sec 
 
 
parameter set:
750
running LASAM ...

********************************************************* 
-------------------- Simulation Summ


********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  45.3668193614 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        = 201.9300000000 cm
Final water in soil       =  43.3138272158 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            =   0.0000000000 cm
GIUH runoff               =   0.0000000000 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  49.8991888714 cm
Total AET                 = 154.0838032764 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       =   0.0000000000 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   -2.253067e-09 cm
Time                      =   1.073 sec 
 
 
parameter set:
760
running LASAM ...

********************************************************* 
-------------------- Simulation Sum


********************************************************* 
-------------------- Simulation Summary ----------------- 
------------------------ Mass balance ------------------- 
Initial water in soil     =  37.4102040920 cm
Total precipitation       = 201.9300000000 cm
Total infiltration        =  58.1835246177 cm
Final water in soil       =  31.2774472874 cm
Surface ponded water      =   0.0000000000 cm
Surface runoff            = 143.7464753823 cm
GIUH runoff               = 143.7464753823 cm
GIUH water (in array)     =   0.0000000000 cm
Total percolation         =  -0.3072957539 cm
Total AET                 =  64.6235772529 cm
Total PET                 = 178.2044578238 cm
Total discharge (Q)       = 143.7464753823 cm
Vol change (calibration)  =   0.0000000000 cm
Global balance            =   -7.668666e-08 cm
Time                      =   2.037 sec 
 
 
parameter set:
770
running LASAM ...

********************************************************* 
-------------------- Simulation Sum

SystemExit: 

/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3445: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
!pwd

In [ ]:
result

In [ ]:
print("elapsed time: ")
print(total_time)
print("num_successful_runs:")
print(num_successful_runs)
print("num_runs:")
print(num_runs)
print("success rate (%): ")
print(100*num_successful_runs/num_runs)
print("unsuccessful_parameter_sets:")
print(unsuccessful_parameter_sets)
print("unsuccessful parameter sets due to erroneous ponded water: ")
print(unsuccessful_parameter_sets_ponded_water)
print("unsuccessful parameter sets due to mass balance error: ")
print(unsuccessful_parameter_sets_mass_bal)
print("unsuccessful parameter sets due to timeout (infinite loop likely): ")
print(unsuccessful_parameter_sets_timeout)

In [ ]:
global_balance_vec_filtered = []
for item in global_balance_vec:
    if (not np.isinf(np.array(item))):
        global_balance_vec_filtered.append(item)
global_balance_vec = global_balance_vec_filtered

In [ ]:
plt.hist(global_balance_vec)

In [ ]:
plt.hist(runtime_vec)

In [ ]:
np.mean(runtime_vec)

In [ ]:
unsuccessful_parameter_sets == unsuccessful_parameter_sets_timeout

In [ ]:
# unsuccessful_parameter_sets

In [ ]:
x[unsuccessful_parameter_sets]

In [ ]:
# result[1]

In [ ]:
# unsuccessful_parameter_sets_ponded_water 

In [ ]:
# unsuccessful_parameter_sets_timeout 

In [ ]:
# unsuccessful_parameter_sets_mass_bal 

In [ ]:
# result[8][0:20]=='Surface ponded water'

In [ ]:
# result

In [ ]:
# !pwd

In [ ]:
plt.hist(global_balance_vec)

In [ ]:
count = 0
for item in global_balance_vec:
    if item>-4:
        count = count + 1
count/len(global_balance_vec)

In [ ]:
###example of how the psi-theta relationship (soil water retention curve) loses its 1:1
###behavior for small psi in the van Genuchten model
###also true for extremely large psi 

# Defining the variables
theta_r = 0.02
theta_s = 0.6
alpha = 0.001
n = 2.8
psi = 0.001

# Calculating the expression
theta = theta_r + (theta_s - theta_r) / (1 + (alpha * psi)**n)**(1 - 1/n)
theta

#note that for psi = 0.001, theta evaluates to exactly theta_s and not slightly less.

In [ ]:
# Defining the variables
theta_r = 0.1
theta_s = 0.4
alpha = 0.1
n = 1.4
psi = 0.001

# Calculating the expression
theta = theta_r + (theta_s - theta_r) / (1 + (alpha * psi)**n)**(1 - 1/n)
theta

###in contrast, note that in this case, the same psi value yields a theta that is less than
###theta_s. While the difference is small, it's enough to cause an infinite WF depth because
##when a WF crosses layer boundaries from the second soil to the first, there will be no delta
###psi that allows for mass balance closure because theta will not change wrt psi 

In [ ]:
#and also here is just another interesting fact, psi can get enormous as theta->theta_r

In [ ]:
import math

# Given values
alpha = 0.18338845850191313
n = 1.0469145
Se = 0.042361

# Calculate m
m = 1 - (1 / n)

# Calculate the expression
result = (1.0 / alpha) * pow(pow(Se, -1.0 / m) - 1.0, 1.0 / n)

# Print the result
print("The result of the expression is:", result)